In [ ]:
# Force downgrade numpy DULU sebelum apapun
import subprocess
subprocess.run(["pip", "install", "numpy>=1.26.0,<2.0.0", "--force-reinstall", "-q"], check=True)

# Restart otomatis
import os
os.kill(os.getpid(), 9)

In [1]:
import numpy as np
print(np.__version__)  # HARUS 1.26.x, bukan 2.x

1.26.4


In [ ]:
# ============================================================
# CELL 1 — Install dependencies
# ============================================================
# CELL 1 — Install (jalankan lalu RESTART RUNTIME setelahnya)
!pip install torch==2.2.2 torchvision==0.17.2 torchaudio==2.2.2 --index-url https://download.pytorch.org/whl/cu121 -q
!pip install "numpy>=1.26.0,<2.0.0" -q
!pip install pandas scikit-learn -q
!pip install timm tqdm opencv-python-headless matplotlib -q
!pip install facenet-pytorch -q
!pip install "ultralytics==8.4.32" -q

# Auto restart kernel setelah install
import os
os.kill(os.getpid(), 9)

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
tobler 0.13.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but

In [2]:

# ============================================================
# CELL 2 — Imports
# ============================================================
import os
import re
import gc
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.transforms as T
import torchvision.transforms.functional as TF
import timm

from PIL import Image
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from ultralytics import YOLO
from facenet_pytorch import MTCNN

In [3]:
# ============================================================
# CELL 3 — Device
# ============================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


Using device: cuda


In [4]:
# ============================================================
# CELL 4 — Load detection models
# ============================================================
yolo_model = YOLO("yolov8n.pt")

retina_model = MTCNN(keep_all=True, device=device)

In [5]:
# ============================================================
# CELL 5 — Mount Google Drive
# ============================================================
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:

# ============================================================
# CELL 6 — Dataset loaders
# ============================================================
def load_train_dataset(train_path, limit=None):
    """
    Load dataset train (multiclass folder).

    Returns:
        image_paths : list of str
        labels      : list of int
        label_to_id : dict  {class_name -> int}
    """
    train_path = Path(train_path)
    class_dirs = sorted([d for d in train_path.iterdir() if d.is_dir()])
    label_to_id = {cls.name: idx for idx, cls in enumerate(class_dirs)}

    image_paths, labels = [], []
    for cls in class_dirs:
        files = (
            list(cls.glob("*.jpg")) +
            list(cls.glob("*.png")) +
            list(cls.glob("*.jpeg"))
        )
        if limit:
            files = files[:limit]
        for p in files:
            image_paths.append(str(p))
            labels.append(label_to_id[cls.name])

    return image_paths, labels, label_to_id


def load_test_dataset(test_path):
    """
    Load dataset test (tanpa label).

    Returns:
        image_paths : list of str
        image_ids   : list of str  (stem filename)
    """
    test_path = Path(test_path)
    files = (
        list(test_path.glob("*.jpg")) +
        list(test_path.glob("*.png")) +
        list(test_path.glob("*.jpeg"))
    )
    image_paths = [str(p) for p in files]
    image_ids   = [p.stem for p in files]
    return image_paths, image_ids


def build_train_df(image_paths, labels):
    return pd.DataFrame({"path": image_paths, "label": labels})


def build_test_df(image_paths, image_ids):
    return pd.DataFrame({"path": image_paths, "id": image_ids})



In [7]:

# ============================================================
# CELL 7 — Load raw dataframes
# ============================================================
TRAIN_PATH = "/content/drive/MyDrive/DATASET/data-analytics-competition-dac-find-it-2026/train"
TEST_PATH  = "/content/drive/MyDrive/DATASET/data-analytics-competition-dac-find-it-2026/test"

train_paths, train_labels, label_map = load_train_dataset(TRAIN_PATH, limit=None)
test_paths,  test_ids               = load_test_dataset(TEST_PATH)

train_df = build_train_df(train_paths, train_labels)
test_df  = build_test_df(test_paths,  test_ids)

print("label_map:", label_map)
print(train_df.head())
print(test_df.head())


label_map: {'fake_mannequin': 0, 'fake_mask': 1, 'fake_printed': 2, 'fake_screen': 3, 'fake_unknown': 4, 'realperson': 5}
                                                path  label
0  /content/drive/MyDrive/DATASET/data-analytics-...      0
1  /content/drive/MyDrive/DATASET/data-analytics-...      0
2  /content/drive/MyDrive/DATASET/data-analytics-...      0
3  /content/drive/MyDrive/DATASET/data-analytics-...      0
4  /content/drive/MyDrive/DATASET/data-analytics-...      0
                                                path        id
0  /content/drive/MyDrive/DATASET/data-analytics-...  test_004
1  /content/drive/MyDrive/DATASET/data-analytics-...  test_005
2  /content/drive/MyDrive/DATASET/data-analytics-...  test_003
3  /content/drive/MyDrive/DATASET/data-analytics-...  test_001
4  /content/drive/MyDrive/DATASET/data-analytics-...  test_008


In [8]:

# ============================================================
# CELL 8 — Face detection helper
# ============================================================
def detect_face_ensemble(
    image_path,
    yolo_model,
    retina_model,
    conf=0.3,
    pad_tight=0.05,
    pad_loose=0.2,
    fallback=True,
):
    img = cv2.imread(image_path)
    if img is None:
        return None, None
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    candidates = []

    # YOLO
    yolo_results = yolo_model(img, conf=conf, verbose=False)[0]
    if len(yolo_results.boxes) > 0:
        for b in yolo_results.boxes.xyxy.cpu().numpy():
            x1, y1, x2, y2 = b
            candidates.append([x1, y1, x2, y2, (x2 - x1) * (y2 - y1)])

    # MTCNN
    boxes, _ = retina_model.detect(img)
    if boxes is not None:
        for b in boxes:
            x1, y1, x2, y2 = b
            candidates.append([x1, y1, x2, y2, (x2 - x1) * (y2 - y1)])

    if len(candidates) == 0:
        if fallback:
            return img, img
        return None, None

    candidates = np.array(candidates)
    x1, y1, x2, y2 = candidates[np.argmax(candidates[:, 4])][:4]
    bw, bh = x2 - x1, y2 - y1

    # tight crop
    tx1 = int(max(0, x1 - pad_tight * bw))
    ty1 = int(max(0, y1 - pad_tight * bh))
    tx2 = int(min(w,  x2 + pad_tight * bw))
    ty2 = int(min(h,  y2 + pad_tight * bh))
    tight = img[ty1:ty2, tx1:tx2]

    # loose crop
    lx1 = int(max(0, x1 - pad_loose * bw))
    ly1 = int(max(0, y1 - pad_loose * bh))
    lx2 = int(min(w,  x2 + pad_loose * bw))
    ly2 = int(min(h,  y2 + pad_loose * bh))
    loose = img[ly1:ly2, lx1:lx2]

    return tight, loose



In [10]:


# ============================================================
# CELL 9 — Process & save face crops
# ============================================================
def process_dataset_dual(df, save_dir, yolo_model, retina_model):
    """
    Deteksi wajah untuk setiap baris df, simpan tight & loose crop.
    Mengembalikan df dengan kolom 'face_tight' dan 'face_loose'.
    """
    os.makedirs(save_dir, exist_ok=True)
    tight_paths, loose_paths = [], []

    for i in tqdm(range(len(df))):
        img_path = df.iloc[i]["path"]
        tight, loose = detect_face_ensemble(img_path, yolo_model, retina_model)

        if tight is None or loose is None:
            tight_paths.append(None)
            loose_paths.append(None)
            continue

        tight_path = os.path.join(save_dir, f"{i}_tight.jpg")
        loose_path = os.path.join(save_dir, f"{i}_loose.jpg")
        cv2.imwrite(tight_path, cv2.cvtColor(tight, cv2.COLOR_RGB2BGR))
        cv2.imwrite(loose_path, cv2.cvtColor(loose, cv2.COLOR_RGB2BGR))

        tight_paths.append(tight_path)
        loose_paths.append(loose_path)

    df = df.copy()
    df["face_tight"] = tight_paths
    df["face_loose"] = loose_paths
    return df


# ---- Jalankan sekali untuk train ----
train_df = process_dataset_dual(
    train_df,
    save_dir="/content/drive/MyDrive/faces/train",
    yolo_model=yolo_model,
    retina_model=retina_model,
)
train_df.to_csv("/content/drive/MyDrive/faces/train_df_processed.csv", index=False)

# ---- Jalankan sekali untuk test ----
test_df = process_dataset_dual(
    test_df,
    save_dir="/content/drive/MyDrive/faces/test",
    yolo_model=yolo_model,
    retina_model=retina_model,
)
test_df.to_csv("/content/drive/MyDrive/faces/test_df_processed.csv", index=False)



100%|██████████| 404/404 [05:30<00:00,  1.22it/s]


In [11]:

# ============================================================
# CELL 10 — Load processed CSVs (session berikutnya)
# ============================================================
train_df = pd.read_csv("/content/drive/MyDrive/faces/train_df_processed.csv")
test_df  = pd.read_csv("/content/drive/MyDrive/faces/test_df_processed.csv")

# Drop baris yang gagal deteksi
train_df = train_df.dropna(subset=["face_tight", "face_loose"]).reset_index(drop=True)
test_df  = test_df.dropna(subset=["face_tight", "face_loose"]).reset_index(drop=True)

print(f"Train rows: {len(train_df)} | Test rows: {len(test_df)}")



Train rows: 1438 | Test rows: 404


In [12]:

# ============================================================
# CELL 11 — Train / Val split
# FIX: split SETELAH face paths sudah ada di kolom,
#      tidak perlu re-assign dari tight_files dict lagi.
# ============================================================
train_df, val_df = train_test_split(
    train_df,
    test_size=0.2,
    stratify=train_df["label"],
    random_state=42,
)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)

print(f"Train: {len(train_df)} | Val: {len(val_df)}")


Train: 1150 | Val: 288


In [13]:

# ============================================================
# CELL 12 — Transforms & Dataset
# ============================================================
train_tf = T.Compose([
    T.Resize((224, 224)),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    T.Lambda(lambda img: img.filter(__import__("PIL").ImageFilter.GaussianBlur(radius=1))),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_tf = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


class FaceDataset(Dataset):
    def __init__(self, df, transform=None, is_train=True):
        self.df        = df.reset_index(drop=True)
        self.transform = transform
        self.is_train  = is_train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        import random
        row = self.df.iloc[idx]

        img_tight = Image.open(row["face_tight"]).convert("RGB")
        img_loose = Image.open(row["face_loose"]).convert("RGB")
        label     = int(row["label"])

        if self.is_train:
            # sinkron horizontal flip
            if random.random() < 0.5:
                img_tight = TF.hflip(img_tight)
                img_loose = TF.hflip(img_loose)
            # sinkron rotasi
            angle = random.uniform(-10, 10)
            img_tight = TF.rotate(img_tight, angle)
            img_loose = TF.rotate(img_loose, angle)

        if self.transform:
            img_tight = self.transform(img_tight)
            img_loose = self.transform(img_loose)

        return img_tight, img_loose, label


class TestFaceDataset(Dataset):
    """Dataset untuk test set (tidak ada label)."""
    def __init__(self, df, transform=None):
        self.df        = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_tight = Image.open(row["face_tight"]).convert("RGB")
        img_loose = Image.open(row["face_loose"]).convert("RGB")
        img_id    = row["id"]

        if self.transform:
            img_tight = self.transform(img_tight)
            img_loose = self.transform(img_loose)

        return img_tight, img_loose, img_id


train_ds = FaceDataset(train_df, transform=train_tf, is_train=True)
val_ds   = FaceDataset(val_df,   transform=val_tf,   is_train=False)
test_ds  = TestFaceDataset(test_df, transform=val_tf)

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=8, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=8, shuffle=False, num_workers=2, pin_memory=True)  # FIX: test_loader dibuat



In [14]:

# ============================================================
# CELL 13 — Model definitions
# ============================================================

# ---------- EfficientNet-B3 ----------
class DualEfficientNet(nn.Module):
    def __init__(self, num_classes=6):
        super().__init__()
        self.backbone = timm.create_model("efficientnet_b3", pretrained=True, num_classes=0)
        in_features   = self.backbone.num_features
        self.classifier = nn.Sequential(
            nn.Linear(in_features * 2, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )

    def forward(self, x_tight, x_loose):
        return self.classifier(torch.cat([self.backbone(x_tight), self.backbone(x_loose)], dim=1))


# ---------- ConvNeXt-Tiny ----------
class DualConvNeXt(nn.Module):
    def __init__(self, num_classes=6):
        super().__init__()
        self.backbone = timm.create_model("convnext_tiny", pretrained=True, num_classes=0)
        self.backbone.set_grad_checkpointing(True)
        in_features   = self.backbone.num_features
        self.classifier = nn.Sequential(
            nn.Linear(in_features * 2, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )

    def forward(self, x_tight, x_loose):
        return self.classifier(torch.cat([self.backbone(x_tight), self.backbone(x_loose)], dim=1))


# ---------- Swin-Tiny (ViT) ----------
class DualViT(nn.Module):
    def __init__(self, num_classes=6):
        super().__init__()
        self.backbone = timm.create_model("swin_tiny_patch4_window7_224", pretrained=True, num_classes=0)
        self.backbone.set_grad_checkpointing(True)
        in_features   = self.backbone.num_features
        self.classifier = nn.Sequential(
            nn.Linear(in_features * 2, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )

    def forward(self, x_tight, x_loose):
        return self.classifier(torch.cat([self.backbone(x_tight), self.backbone(x_loose)], dim=1))



In [17]:

# ============================================================
# CELL 14 — Generic train function (dipakai semua model)
# FIX: GradScaler & autocast pakai API baru PyTorch 2.x
# FIX: semua hyperparameter (optimizer, scheduler) di dalam fungsi
# ============================================================
def train_model(
    model,
    train_loader,
    val_loader,
    device,
    epochs=10,
    lr=1e-4,
    save_path="best_model.pth",
    model_name="Model",
):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    # FIX: PyTorch 2.2.2 masih pakai torch.cuda.amp
    use_amp = device.type == "cuda"
    scaler  = torch.cuda.amp.GradScaler(enabled=use_amp)

    best_acc = 0.0

    for epoch in range(epochs):
        torch.cuda.empty_cache()
        gc.collect()

        print(f"\n[{model_name}] Epoch {epoch+1}/{epochs}")

        # ===== TRAIN =====
        model.train()
        train_loss = 0.0

        for img_tight, img_loose, labels in train_loader:
            img_tight = img_tight.to(device, non_blocking=True)
            img_loose = img_loose.to(device, non_blocking=True)
            labels    = labels.to(device, non_blocking=True)

            optimizer.zero_grad()

            # FIX: torch.cuda.amp.autocast untuk PyTorch 2.2.2
            with torch.cuda.amp.autocast(enabled=use_amp):
                outputs = model(img_tight, img_loose)
                loss    = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            train_loss += loss.item()
            del img_tight, img_loose, labels, outputs, loss

        # ===== VALIDATION =====
        model.eval()
        val_loss = 0.0
        correct  = 0
        total    = 0

        with torch.no_grad():
            for img_tight, img_loose, labels in val_loader:
                img_tight = img_tight.to(device, non_blocking=True)
                img_loose = img_loose.to(device, non_blocking=True)
                labels    = labels.to(device, non_blocking=True)

                with torch.cuda.amp.autocast(enabled=use_amp):
                    outputs = model(img_tight, img_loose)
                    loss    = criterion(outputs, labels)

                val_loss += loss.item()
                pred      = outputs.argmax(1)
                correct  += (pred == labels).sum().item()
                total    += labels.size(0)

                del img_tight, img_loose, labels, outputs, loss

        val_acc = correct / total
        print(f"  Train Loss: {train_loss/len(train_loader):.4f}")
        print(f"  Val Loss:   {val_loss/len(val_loader):.4f}  |  Val Acc: {val_acc:.4f}")

        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), save_path)
            print(f"  ✓ Best model saved → {save_path}  (acc={best_acc:.4f})")

        scheduler.step()

    print(f"\n[{model_name}] Training selesai. Best Val Acc: {best_acc:.4f}")
    return model

In [18]:

# ============================================================
# CELL 15 — Train EfficientNet
# ============================================================
model_eff = DualEfficientNet(num_classes=6).to(device)
model_eff = train_model(
    model_eff, train_loader, val_loader,
    device=device, epochs=10, lr=1e-4,
    save_path="/content/best_model.pth",
    model_name="EfficientNet-B3",
)



[EfficientNet-B3] Epoch 1/10
  Train Loss: 1.1430
  Val Loss:   nan  |  Val Acc: 0.4722
  ✓ Best model saved → /content/best_model.pth  (acc=0.4722)

[EfficientNet-B3] Epoch 2/10
  Train Loss: 0.4584
  Val Loss:   nan  |  Val Acc: 0.5660
  ✓ Best model saved → /content/best_model.pth  (acc=0.5660)

[EfficientNet-B3] Epoch 3/10
  Train Loss: 0.2339
  Val Loss:   nan  |  Val Acc: 0.5278

[EfficientNet-B3] Epoch 4/10
  Train Loss: 0.1718
  Val Loss:   nan  |  Val Acc: 0.6424
  ✓ Best model saved → /content/best_model.pth  (acc=0.6424)

[EfficientNet-B3] Epoch 5/10
  Train Loss: 0.1142
  Val Loss:   nan  |  Val Acc: 0.7431
  ✓ Best model saved → /content/best_model.pth  (acc=0.7431)

[EfficientNet-B3] Epoch 6/10
  Train Loss: 0.0897
  Val Loss:   nan  |  Val Acc: 0.6493

[EfficientNet-B3] Epoch 7/10
  Train Loss: 0.0859
  Val Loss:   nan  |  Val Acc: 0.6597

[EfficientNet-B3] Epoch 8/10
  Train Loss: 0.0603
  Val Loss:   nan  |  Val Acc: 0.6667

[EfficientNet-B3] Epoch 9/10
  Train Loss: 

In [20]:

# ============================================================
# CELL 16 — Train ConvNeXt
# ============================================================
model_cnx = DualConvNeXt(num_classes=6).to(device)
model_cnx = train_model(
    model_cnx, train_loader, val_loader,
    device=device, epochs=10, lr=1e-4,
    save_path="/content/convnext_best.pth",
    model_name="ConvNeXt-Tiny",
)



[ConvNeXt-Tiny] Epoch 1/10
  Train Loss: 0.8939
  Val Loss:   0.4921  |  Val Acc: 0.8542
  ✓ Best model saved → /content/convnext_best.pth  (acc=0.8542)

[ConvNeXt-Tiny] Epoch 2/10
  Train Loss: 0.3910
  Val Loss:   0.4609  |  Val Acc: 0.8403

[ConvNeXt-Tiny] Epoch 3/10
  Train Loss: 0.2527
  Val Loss:   0.2542  |  Val Acc: 0.9306
  ✓ Best model saved → /content/convnext_best.pth  (acc=0.9306)

[ConvNeXt-Tiny] Epoch 4/10
  Train Loss: 0.1398
  Val Loss:   0.3397  |  Val Acc: 0.8854

[ConvNeXt-Tiny] Epoch 5/10
  Train Loss: 0.0770
  Val Loss:   0.1841  |  Val Acc: 0.9340
  ✓ Best model saved → /content/convnext_best.pth  (acc=0.9340)

[ConvNeXt-Tiny] Epoch 6/10
  Train Loss: 0.0233
  Val Loss:   0.1557  |  Val Acc: 0.9514
  ✓ Best model saved → /content/convnext_best.pth  (acc=0.9514)

[ConvNeXt-Tiny] Epoch 7/10
  Train Loss: 0.0086
  Val Loss:   0.1295  |  Val Acc: 0.9549
  ✓ Best model saved → /content/convnext_best.pth  (acc=0.9549)

[ConvNeXt-Tiny] Epoch 8/10
  Train Loss: 0.0039
 

In [21]:

# ============================================================
# CELL 17 — Train ViT (Swin-Tiny)
# ============================================================
model_vit = DualViT(num_classes=6).to(device)
model_vit = train_model(
    model_vit, train_loader, val_loader,
    device=device, epochs=10, lr=3e-5,
    save_path="/content/vit_best.pth",
    model_name="Swin-Tiny",
)


model.safetensors:   0%|          | 0.00/114M [00:00<?, ?B/s]


[Swin-Tiny] Epoch 1/10
  Train Loss: 0.9651
  Val Loss:   0.3908  |  Val Acc: 0.8785
  ✓ Best model saved → /content/vit_best.pth  (acc=0.8785)

[Swin-Tiny] Epoch 2/10
  Train Loss: 0.3375
  Val Loss:   0.4300  |  Val Acc: 0.8542

[Swin-Tiny] Epoch 3/10
  Train Loss: 0.2047
  Val Loss:   0.2218  |  Val Acc: 0.9340
  ✓ Best model saved → /content/vit_best.pth  (acc=0.9340)

[Swin-Tiny] Epoch 4/10
  Train Loss: 0.1065
  Val Loss:   0.1918  |  Val Acc: 0.9583
  ✓ Best model saved → /content/vit_best.pth  (acc=0.9583)

[Swin-Tiny] Epoch 5/10
  Train Loss: 0.0511
  Val Loss:   0.1727  |  Val Acc: 0.9583

[Swin-Tiny] Epoch 6/10
  Train Loss: 0.0386
  Val Loss:   0.2144  |  Val Acc: 0.9479

[Swin-Tiny] Epoch 7/10
  Train Loss: 0.0261
  Val Loss:   0.1888  |  Val Acc: 0.9549

[Swin-Tiny] Epoch 8/10
  Train Loss: 0.0181
  Val Loss:   0.2094  |  Val Acc: 0.9549

[Swin-Tiny] Epoch 9/10
  Train Loss: 0.0152
  Val Loss:   0.1991  |  Val Acc: 0.9479

[Swin-Tiny] Epoch 10/10
  Train Loss: 0.0141
  V

In [22]:
# ============================================================
# CELL 18 — Load best weights untuk ensemble
# ============================================================
model1 = DualEfficientNet(num_classes=6)
model1.load_state_dict(torch.load("/content/best_model.pth",     map_location=device))
model1.to(device).eval()

model2 = DualConvNeXt(num_classes=6)
model2.load_state_dict(torch.load("/content/convnext_best.pth", map_location=device))
model2.to(device).eval()

model3 = DualViT(num_classes=6)
model3.load_state_dict(torch.load("/content/vit_best.pth",      map_location=device))
model3.to(device).eval()


DualViT(
  (backbone): SwinTransformer(
    (patch_embed): PatchEmbed(
      (proj): Conv2d(3, 96, kernel_size=(4, 4), stride=(4, 4))
      (norm): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
    )
    (layers): Sequential(
      (0): SwinTransformerStage(
        (downsample): Identity()
        (blocks): Sequential(
          (0): SwinTransformerBlock(
            (norm1): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
            (attn): WindowAttention(
              (qkv): Linear(in_features=96, out_features=288, bias=True)
              (attn_drop): Dropout(p=0.0, inplace=False)
              (proj): Linear(in_features=96, out_features=96, bias=True)
              (proj_drop): Dropout(p=0.0, inplace=False)
              (softmax): Softmax(dim=-1)
            )
            (drop_path1): Identity()
            (norm2): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
            (mlp): Mlp(
              (fc1): Linear(in_features=96, out_features=384, bias=True)


In [23]:
# ============================================================
# CELL 19 — Inference & Ensemble
# FIX: test_loader sekarang ada (dibuat di CELL 12)
# ============================================================
preds_list = []
ids_list   = []

with torch.no_grad():
    for img_tight, img_loose, img_ids in test_loader:
        img_tight = img_tight.to(device)
        img_loose = img_loose.to(device)

        p1 = F.softmax(model1(img_tight, img_loose), dim=1)
        p2 = F.softmax(model2(img_tight, img_loose), dim=1)
        p3 = F.softmax(model3(img_tight, img_loose), dim=1)

        prob = (p1 + p2 + p3) / 3
        pred = prob.argmax(1).cpu().numpy()

        preds_list.extend(pred.tolist())
        ids_list.extend(list(img_ids))


In [24]:

# ============================================================
# CELL 20 — Buat submission CSV
# ============================================================
id_to_label  = {v: k for k, v in label_map.items()}
pred_labels  = [id_to_label[p] for p in preds_list]

submission = pd.DataFrame({"id": ids_list, "label": pred_labels})
submission = submission.sort_values("id").reset_index(drop=True)
submission.to_csv("submission_ensemble.csv", index=False)

print(submission.head())
print(f"Shape: {submission.shape}")
print(f"Unique labels: {submission['label'].unique()}")

         id           label
0  test_001     fake_screen
1  test_002  fake_mannequin
2  test_003      realperson
3  test_004      realperson
4  test_005    fake_printed
Shape: (404, 2)
Unique labels: ['fake_screen' 'fake_mannequin' 'realperson' 'fake_printed' 'fake_mask' 'fake_unknown']


# Cek Ground Truth

In [26]:
import pandas as pd

sub = pd.read_csv("/content/submission_ensemble.csv")      # hasil prediksi
gt  = pd.read_csv("/content/drive/MyDrive/DATASET/ground_truth.csv")    # label benar

In [27]:
print(sub.head())
print(gt.head())

         id           label
0  test_001     fake_screen
1  test_002  fake_mannequin
2  test_003      realperson
3  test_004      realperson
4  test_005    fake_printed
         id           label  Unnamed: 2 Unnamed: 3  Unnamed: 4
0  test_001     fake_screen         NaN        NaN         NaN
1  test_002  fake_mannequin         NaN        NaN         NaN
2  test_003      realperson         NaN        NaN         1.0
3  test_004      realperson         NaN        NaN         NaN
4  test_005    fake_printed         NaN        NaN         NaN


In [28]:
gt = gt[['id', 'label']]

In [29]:
df_eval = sub.merge(gt, on="id", suffixes=("_pred", "_true"))

In [30]:
acc = (df_eval["label_pred"] == df_eval["label_true"]).mean()
print("Accuracy:", acc)

Accuracy: 0.8861386138613861


Catatan :
1. crop Yolo, ensemble Efficientnet_convnext_ViT dengan soft Voting 1/3
Accuracy: 0.9207920792079208

2. crop Yolo, ensemble Efficientnet_convnext_ViT dengan Weight 2,4,4
Accuracy: 0.9207920792079208

3. crop Yolo, ensemble Efficientnet_convnext_ViT dengan Weight 2, 4.5 , 3.5
(Accuracy: 0.91)

4. crop Yolo, ensemble Efficientnet_convnext_ViT dengan Weight 2, 3 , 5
(Accuracy: 0.9232673267326733)

5. 226 sama

6. 118 Accuracy: 0.905940594059406

7. yolo + reitinaface sama seperti 118

8.

In [31]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    df_eval["label_true"],
    df_eval["label_pred"]
)

print(cm)

[[45  0  0  0  0  0]
 [ 6 63  0  0  0  1]
 [ 2  2 51  0  3  5]
 [ 1  6  1 57  4  4]
 [ 3  0  1  0 45  0]
 [ 0  7  0  0  0 97]]


In [32]:
from sklearn.metrics import classification_report

print(classification_report(
    df_eval["label_true"],
    df_eval["label_pred"]
))

                precision    recall  f1-score   support

fake_mannequin       0.79      1.00      0.88        45
     fake_mask       0.81      0.90      0.85        70
  fake_printed       0.96      0.81      0.88        63
   fake_screen       1.00      0.78      0.88        73
  fake_unknown       0.87      0.92      0.89        49
    realperson       0.91      0.93      0.92       104

      accuracy                           0.89       404
     macro avg       0.89      0.89      0.88       404
  weighted avg       0.90      0.89      0.89       404



In [33]:
wrong = df_eval[df_eval["label_pred"] != df_eval["label_true"]]
print(wrong.head())
print(wrong.shape)
wrong.to_csv("wrong.csv", index=False)

          id    label_pred    label_true
6   test_007     fake_mask    realperson
13  test_014     fake_mask   fake_screen
23  test_024  fake_unknown   fake_screen
25  test_026  fake_printed   fake_screen
30  test_031    realperson  fake_printed
(46, 3)


In [34]:
# Jalankan ini untuk lihat distribusi data
print("=== Distribusi Train ===")
print(train_df["label"].value_counts().sort_index())
print(f"\nTotal train: {len(train_df)}")
print(f"Total val  : {len(val_df)}")
print(f"Jumlah class: {train_df['label'].nunique()}")

=== Distribusi Train ===
label
0    157
1    201
2     77
3    150
4    238
5    327
Name: count, dtype: int64

Total train: 1150
Total val  : 288
Jumlah class: 6
